In [1]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

In [2]:
# Define image transformations (Resize to 64x64 and convert to tensor)
transform = transforms.Compose([
    transforms.Resize((64, 64)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5], std=[0.5])
])

In [5]:
# Correct dataset path for Casting defect detection based on workspace
data_dir = '../data/casting_data'

In [6]:
# Load train dataset using ImageFolder
if os.path.exists(data_dir):
    train_dataset = datasets.ImageFolder(root=os.path.join(data_dir, 'train'), transform=transform)
    train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
    print("Casting Train Dataset Loaded Successfully!")
    print("Classes found:", train_dataset.classes)
    print("Total training images:", len(train_dataset))
else:
    print(f"Path not found: {data_dir}. Please check your folder structure.")

Casting Train Dataset Loaded Successfully!
Classes found: ['defective', 'normal']
Total training images: 6633


In [7]:
# Define CNN Model for Image Classification
class Casting_CNN(nn.Module):
    def __init__(self):
        super(Casting_CNN, self).__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 16, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
            
            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
            
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2)
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * 8 * 8, 128),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(128, 2)  # 2 classes: defective, normal
        )
        
    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

In [8]:
# Initialize model, loss, and optimizer
cnn_model = Casting_CNN()
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(cnn_model.parameters(), lr=0.001)

In [9]:
# Training Loop
epochs = 3
cnn_model.train()

for epoch in range(epochs):
    running_loss = 0.0
    for images, labels in train_loader:
        optimizer.zero_grad()
        outputs = cnn_model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * images.size(0)
        
    epoch_loss = running_loss / len(train_loader.dataset)
    print(f"Epoch [{epoch+1}/{epochs}], Loss: {epoch_loss:.4f}")

print("Casting CNN Model Training Completed Successfully!")

Epoch [1/3], Loss: 0.3595
Epoch [2/3], Loss: 0.1871
Epoch [3/3], Loss: 0.1179
Casting CNN Model Training Completed Successfully!


In [10]:
import os

# Save the trained CNN model state dict
torch.save(cnn_model.state_dict(), '../saved_models/casting_cnn_model.pth')
print("Casting CNN Model saved successfully inside 'saved_models' folder!")

Casting CNN Model saved successfully inside 'saved_models' folder!
